# Phase 2: # Phase 2: Feature Relevance for Demand Forecasting & Safety Stock

This section reviews all columns for their relevance to demand forecasting and safety stock calculation. It documents which features are included or excluded, and the rationale for each decision.

## 1. Import Required Libraries
Import pandas, numpy, matplotlib, seaborn, and any other libraries needed for data analysis and visualization.

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set(style="whitegrid")

## 2. Load Dataset

In [2]:

df = pd.read_csv('/home/marc/project/supply_chain_DS/kaggle_supply_chain_dataSet/supply_chain_data.csv')
df.head()

,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,Mumbai,29,215,29,46.279879,Pending,0.226410,Road,Route B,187.752075
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,Mumbai,23,517,30,33.616769,Pending,4.854068,Road,Route B,503.065579
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,Mumbai,12,971,27,30.688019,Pending,4.580593,Air,Route C,141.920282
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,Kolkata,24,937,18,35.624741,Fail,4.746649,Rail,Route A,254.776159
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,Delhi,5,414,3,92.065161,Fail,3.145580,Air,Route A,923.440632


In [7]:
df.shape

(100, 24)

## 3. Feature Relevance Table

The following table summarizes the relevance of each column for demand forecasting and safety stock calculation. Columns are marked as 'Include' or 'Exclude' with rationale.

In [5]:
# Define relevance for each column
relevance = {
    'Product Type': ('Include', 'Product category, useful for grouping and seasonality'),
    'SKU': ('Include', 'Unique identifier for each product, essential for SKU-level analysis'),
    'Price': ('Include', 'May influence demand'),
    'Availability': ('Include', 'Directly impacts ability to fulfill demand'),
    'Number of products sold': ('Include', 'Target variable for demand forecasting'),
    'Revenue generated': ('Include', 'Correlated with demand, useful for business KPIs'),
    'Customer demographics': ('Exclude', 'Not directly relevant unless granular and available per order'),
    'Stock levels': ('Include', 'Key for safety stock calculation'),
    'Lead times': ('Include', 'Critical for safety stock and replenishment'),
    'Order quantities': ('Include', 'Related to demand and inventory flow'),
    'Shipping times': ('Include', 'Affects lead time and service level'),
    'Shipping carriers': ('Include', 'May impact shipping reliability and lead time'),
    'Shipping costs': ('Include', 'May affect order size and profitability'),
    'Supplier name': ('Include', 'Supplier reliability can affect lead time and stockouts'),
    'Location': ('Include', 'May affect demand and logistics'),
    'Production volumes': ('Include', 'Affects supply capability'),
    'Manufacturing lead time': ('Include', 'Impacts replenishment and safety stock'),
    'Manufacturing costs': ('Include', 'Relevant for profitability, less so for demand'),
    'Inspection results': ('Include', 'Quality issues can affect available stock'),
    'Defect rates': ('Include', 'Affects usable inventory'),
    'Transportation modes': ('Include', 'Impacts lead time and cost'),
    'Routes': ('Include', 'May affect lead time and reliability'),
    'Costs': ('Include', 'General cost information, useful for business KPIs'),
}

relevance_summary = pd.DataFrame([
    {
        'Column': col,
        'Relevance': relevance.get(col, ('Review', 'Needs further assessment'))[0],
        'Rationale': relevance.get(col, ('Review', 'Needs further assessment'))[1]
    }
    for col in df.columns
])
display(relevance_summary)

,Column,Relevance,Rationale
0,Product type,Review,Needs further assessment
1,SKU,Include,"Unique identifier for each product, essential ..."
2,Price,Include,May influence demand
3,Availability,Include,Directly impacts ability to fulfill demand
4,Number of products sold,Include,Target variable for demand forecasting
5,Revenue generated,Include,"Correlated with demand, useful for business KPIs"
6,Customer demographics,Exclude,Not directly relevant unless granular and avai...
7,Stock levels,Include,Key for safety stock calculation
8,Lead times,Include,Critical for safety stock and replenishment
9,Order quantities,Include,Related to demand and inventory flow


---

**Summary:**
- All relevant columns for demand forecasting and safety stock have been identified.
- Irrelevant columns (e.g., Customer demographics) are excluded unless further granularity is available.
- The rationale for each inclusion/exclusion is documented in the table above.

Proceed to feature engineering based on the selected columns.

## 4. Feature Engineering


### Proposed Engineered Features

The following features are engineered to enhance demand forecasting and safety stock analysis:

- **Sales Velocity:** Products sold per lead time unit.
- **Lead Time Variability:** Absolute difference between 'Lead times' and 'Manufacturing lead time'.
- **Stockout Risk:** Ratio of stock levels to order quantities.
- **Revenue per Unit Sold:** Revenue generated divided by number of products sold.
- **Defect Rate Category:** Binned defect rates (Low/Medium/High).
- **Supplier Reliability:** Binary indicator based on inspection results and defect rates.

These features are for exploration and methodology prototyping, given the small dataset size.

In [8]:
# Feature Engineering Implementation

# 1. Sales velocity (products sold per lead time unit)
df['sales_velocity'] = df['Number of products sold'] / df['Lead times']

# 2. Lead time variability (if both columns exist)
if 'Lead times' in df.columns and 'Manufacturing lead time' in df.columns:
    df['lead_time_variability'] = abs(df['Lead times'] - df['Manufacturing lead time'])

# 3. Stockout risk (stock levels to order quantities)
df['stockout_risk'] = df['Stock levels'] / df['Order quantities']

# 4. Revenue per unit sold
df['revenue_per_unit'] = df['Revenue generated'] / df['Number of products sold']

# 5. Defect rate category
df['defect_rate_category'] = pd.cut(df['Defect rates'], bins=[-float('inf'), 1, 3, float('inf')], labels=['Low', 'Medium', 'High'])

# 6. Supplier reliability (example: 1 if inspection pending and defect rate low, else 0)
df['supplier_reliable'] = ((df['Inspection results'] == 'Pending') & (df['Defect rates'] < 1)).astype(int)

df.head()

,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,...,Defect rates,Transportation modes,Routes,Costs,sales_velocity,lead_time_variability,stockout_risk,revenue_per_unit,defect_rate_category,supplier_reliable
0,haircare,SKU0,69.808006,55,802,8661.996792,Non-binary,58,7,96,...,0.226410,Road,Route B,187.752075,114.571429,22,0.604167,10.800495,Low,1
1,skincare,SKU1,14.843523,95,736,7460.900065,Female,53,30,37,...,4.854068,Road,Route B,503.065579,24.533333,0,1.432432,10.137092,High,0
2,haircare,SKU2,11.319683,34,8,9577.749626,Unknown,1,10,88,...,4.580593,Air,Route C,141.920282,0.800000,17,0.011364,1197.218703,High,0
3,skincare,SKU3,61.163343,68,83,7766.836426,Non-binary,23,13,59,...,4.746649,Rail,Route A,254.776159,6.384615,5,0.389831,93.576342,High,0
4,skincare,SKU4,4.805496,26,871,2686.505152,Non-binary,5,3,56,...,3.145580,Air,Route A,923.440632,290.333333,0,0.089286,3.084392,High,0


## 5. Business questions

Here are relevant business questions that can be explored using the newly engineered features:

1. Which SKUs have the highest and lowest sales velocity?
→ Helps identify fast- and slow-moving products for inventory optimization.

2. Are there SKUs or suppliers with high lead time variability?
→ Pinpoints supply chain instability and potential risk areas.

3. Which products are at greatest risk of stockouts?
→ Use the stockout risk feature to prioritize replenishment and safety stock.

3. How does revenue per unit sold vary by product type or supplier?
→ Identifies the most profitable products and partners.

4. Are there patterns between defect rate categories and supplier reliability?
→ Supports supplier evaluation and quality improvement initiatives.

5. Do reliable suppliers (as defined) correlate with lower stockout risk or higher sales velocity?
→ Assesses the impact of supplier quality on supply chain performance.

6. How do engineered features (e.g., sales velocity, stockout risk) change over time or by location?
→ Reveals trends and regional differences for targeted actions.

7. Can we segment products or suppliers based on these features for differentiated inventory policies?
→ Enables tailored strategies for different product/supplier groups.

These questions help drive actionable insights for demand planning, inventory management, and supplier performance improvement.